# KB Enrichment

Three goals:

1. Analyse the combined KB — sources, categories, lengths
2. Prepare a small labelled training set for the future classifier
3. Save the enriched KB with derived features

Input: `data/processed/kb_combined.csv`  
Outputs:
- `data/processed/kb_enriched.csv`
- `data/processed/classifier_train.csv`

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"

kb = pd.read_csv(PROCESSED_DIR / "kb_combined.csv")
print(kb.shape)
print(kb.columns.tolist())

(256841, 7)
['doc_id', 'source', 'source_id', 'category', 'question', 'context', 'answer']


C:\Users\Matrix\AppData\Local\Temp\ipykernel_19820\4234780445.py:4: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  kb = pd.read_csv(PROCESSED_DIR / "kb_combined.csv")


In [3]:
kb.head(3)

,doc_id,source,source_id,category,question,context,answer
0,0,medquad,3005,treatment,What are the treatments for Breast Cancer?,NaN,Key Points - There are different types of trea...
1,1,medquad,3037,treatment,What are the treatments for Childhood Soft Tis...,NaN,Key Points - There are different types of trea...
2,2,medquad,2827,information,What is (are) Childhood Vascular Tumors?,NaN,Key Points - Childhood vascular tumors form fr...


## 1. Composition analysis

Which sources carry the KB, and which of them already have categories?

In [4]:
kb["source"].value_counts()

source
pubmedqa       211267
medredqa        29581
medquad         14978
medlineplus      1015
Name: count, dtype: int64

In [5]:
# Only MedQuAD has a populated category field. Everything else is blank.
kb["has_category"] = kb["category"].fillna("").astype(str).str.strip() != ""

kb.groupby("source")["has_category"].agg(["sum", "count", "mean"]).round(3)

,sum,count,mean
source,,,
medlineplus,0,1015,0.0
medquad,14978,14978,1.0
medredqa,0,29581,0.0
pubmedqa,0,211267,0.0


In [6]:
# What categories does MedQuAD actually provide?
kb.loc[kb["source"] == "medquad", "category"].value_counts()

category
information        3821
symptoms           2683
treatment          2213
inheritance        1382
frequency          1120
genetic changes    1087
causes              657
exams and tests     611
research            369
outlook             354
susceptibility      245
prevention          183
considerations      133
stages               76
complications        43
support groups        1
Name: count, dtype: int64

## 2. Length features

Word counts for question, context, and answer. Used later for chunking and to spot odd records.

In [7]:
for col in ["question", "context", "answer"]:
    kb[f"{col}_words"] = kb[col].fillna("").astype(str).str.split().str.len()

kb[["question_words", "context_words", "answer_words"]].describe().round(1)

,question_words,context_words,answer_words
count,256841.0,256841.0,256841.0
mean,14.1,188.0,51.9
std,5.0,91.0,82.0
min,1.0,0.0,1.0
25%,11.0,155.0,26.0
50%,14.0,194.0,36.0
75%,17.0,224.0,52.0
max,102.0,4518.0,4281.0


In [8]:
# Median length per source
kb.groupby("source")[["question_words", "context_words", "answer_words"]].median().round(0)

,question_words,context_words,answer_words
source,,,
medlineplus,4.0,0.0,193.0
medquad,7.0,0.0,141.0
medredqa,8.0,153.0,47.0
pubmedqa,15.0,200.0,34.0


## 3. Classifier training set

MedQuAD is the only source with real category labels. We map its native
`qtype` values onto the 5 categories we care about. MedQuAD does not have a
`Medication` label, so the classifier will be trained on 5 classes only.

Mapping used:

| MedQuAD qtype | Target category |
|---------------|-----------------|
| symptoms | Symptoms |
| exams and tests | Diagnosis |
| treatment | Treatment |
| susceptibility | Prevention |
| information, causes, outlook, considerations | General |

Genetic-specific qtypes (`inheritance`, `frequency`, `genetic changes`) and `research` are dropped — too specialised for a general consumer assistant.

In [9]:
QTYPE_MAP = {
    "symptoms": "Symptoms",
    "exams and tests": "Diagnosis",
    "stages": "Diagnosis",
    "treatment": "Treatment",
    "complications": "General",
    "susceptibility": "Prevention",
    "prevention": "Prevention",
    "information": "General",
    "causes": "General",
    "outlook": "General",
    "considerations": "General",
    "support groups": "General",
}

mq = kb[kb["source"] == "medquad"].copy()
mq["label"] = mq["category"].str.strip().map(QTYPE_MAP)
mq = mq.dropna(subset=["label"])

print("usable:", len(mq), "/", len(kb[kb["source"] == "medquad"]))
print(mq["label"].value_counts())

usable: 11020 / 14978
label
General       5009
Symptoms      2683
Treatment     2213
Diagnosis      687
Prevention     428
Name: count, dtype: int64


In [10]:
classifier_train = mq[["question", "label"]].reset_index(drop=True)
print(classifier_train.shape)
classifier_train.head(3)

(11020, 2)


,question,label
0,What are the treatments for Breast Cancer?,Treatment
1,What are the treatments for Childhood Soft Tis...,Treatment
2,What is (are) Childhood Vascular Tumors?,General


In [11]:
classifier_train.to_csv(PROCESSED_DIR / "classifier_train.csv", index=False)
print("saved")

saved


## 4. Save enriched KB

In [12]:
kb.to_csv(PROCESSED_DIR / "kb_enriched.csv", index=False)
print("kb_enriched.csv:", kb.shape)

kb_enriched.csv: (256841, 11)


Next: `04_eda.ipynb`